In [1]:
import numpy as np
import pandas as pd
import polars as pl
import torch


import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from scipy.stats import zscore
from tqdm import tqdm
from functools import partial


# home-grown
import utils as ut
import model as mod
import lazydata as lzdt

In [2]:
pl.Config.set_tbl_rows(110)      # default: 25
pl.Config.set_tbl_cols(100)      # default: 10
pl.Config.set_tbl_width_chars(200)

polars.config.Config

In [3]:
%load_ext autoreload
%autoreload 2

In [4]:
df_mm, dict_colnames = ut.load_mm_dataset("full")

In [28]:
# n_participants_total = df_mm.select(pl.col("sid").unique()).shape[0]
n_participants_total = (
    len(df_mm[dict_colnames["col_pid"][0]].unique())
)

in_dim = len(dict_colnames["cols_x"]) + len(dict_colnames["col_y_shifted"])

dict_info = {
    "n_participants_total": n_participants_total,
    "n_trials_train": dict_colnames["n_trials_train"],
    "col_pid": dict_colnames["col_pid"],
    "cols_x": dict_colnames["cols_x"],
    "col_y": dict_colnames["col_y"],
    "col_y_shifted": dict_colnames["col_y_shifted"],
    "l_colnames_single_zscale": dict_colnames["l_colnames_single_zscale"],
    "in_dim": in_dim,
}

In [29]:
df_mm[dict_info["l_colnames_single_zscale"]] = df_mm[dict_info["l_colnames_single_zscale"]].apply(zscore)

In [30]:
df_use = ut.shift_y(df_mm)

In [32]:
df_train, df_dev = ut.train_dev_split(
    df_use,
    splittype="first_vs_second_half",
    n_trial_split=dict_info["n_trials_train"],
)

## Predict using sklearn Boosted Classifier

In [37]:
df_test = df_dev

In [33]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

In [34]:
df_train.shape

(218690, 57)

In [35]:
df_train.head()

,sid,trial_id,left_intervention,PedPed,left_crossingsignal,left_numberofcharacters,LeftHand_left,left_man,left_woman,left_pregnant,...,right_maleathlete,right_femaledoctor,right_maledoctor,right_dog,right_cat,default_right,default_left,sid_unique,right_picked_prev,is_train
0,1002214337,1.0,1,1,0.468807,0.623271,1,-0.518938,-0.519485,-0.237716,...,-0.373072,-0.296139,-0.29594,-0.311612,-0.309982,1,0,1002214337,0,True
1,1002214337,2.0,1,1,-0.750774,1.304685,1,-0.518938,-0.519485,-0.237716,...,-0.373072,-0.296139,-0.29594,-0.311612,-0.309982,1,0,1002214337,0,True
2,1002214337,3.0,0,1,-0.750774,0.623271,1,-0.518938,-0.519485,-0.237716,...,-0.373072,2.668483,-0.29594,-0.311612,-0.309982,0,1,1002214337,0,True
3,1002214337,4.0,0,1,1.688388,0.623271,1,-0.518938,1.173773,-0.237716,...,-0.373072,-0.296139,-0.29594,1.434933,4.889767,0,1,1002214337,1,True
4,1002214337,5.0,1,1,-0.750774,-0.739557,1,-0.518938,-0.519485,-0.237716,...,-0.373072,-0.296139,-0.29594,3.181478,-0.309982,1,0,1002214337,1,True


In [38]:
df_test.shape

(108136, 57)

In [39]:
df_test.head(10)

,sid,trial_id,left_intervention,PedPed,left_crossingsignal,left_numberofcharacters,LeftHand_left,left_man,left_woman,left_pregnant,...,right_maleathlete,right_femaledoctor,right_maledoctor,right_dog,right_cat,default_right,default_left,sid_unique,right_picked_prev,is_train
38,1002214337,39.0,0,0,-0.750774,-1.420971,1,-0.518938,-0.519485,-0.237716,...,-0.373072,-0.296139,-0.29594,-0.311612,-0.309982,0,1,1002214337,1,False
39,1002214337,40.0,1,0,-0.750774,-0.058143,1,-0.518938,-0.519485,-0.237716,...,-0.373072,-0.296139,-0.29594,1.434933,-0.309982,0,0,1002214337,1,False
40,1002214337,41.0,1,0,-0.750774,-0.739557,1,-0.518938,-0.519485,-0.237716,...,1.511147,-0.296139,-0.29594,-0.311612,-0.309982,1,0,1002214337,0,False
41,1002214337,42.0,0,0,0.468807,-1.420971,1,1.171206,-0.519485,-0.237716,...,1.511147,-0.296139,-0.29594,-0.311612,-0.309982,0,1,1002214337,0,False
42,1002214337,43.0,1,1,0.468807,1.304685,1,-0.518938,-0.519485,3.590027,...,1.511147,-0.296139,-0.29594,-0.311612,-0.309982,1,0,1002214337,0,False
43,1002214337,44.0,1,1,1.688388,-0.739557,1,-0.518938,-0.519485,-0.237716,...,-0.373072,-0.296139,-0.29594,-0.311612,-0.309982,1,0,1002214337,0,False
44,1002214337,45.0,0,0,0.468807,1.304685,1,-0.518938,-0.519485,-0.237716,...,-0.373072,-0.296139,-0.29594,-0.311612,-0.309982,0,1,1002214337,1,False
45,1002214337,46.0,1,1,1.688388,-1.420971,1,-0.518938,-0.519485,-0.237716,...,-0.373072,-0.296139,-0.29594,1.434933,-0.309982,1,0,1002214337,0,False
46,1002214337,47.0,0,0,-0.750774,-0.058143,1,-0.518938,-0.519485,-0.237716,...,-0.373072,-0.296139,-0.29594,-0.311612,-0.309982,0,1,1002214337,0,False
47,1002214337,48.0,0,1,-0.750774,0.623271,1,-0.518938,-0.519485,-0.237716,...,1.511147,-0.296139,-0.29594,-0.311612,-0.309982,0,1,1002214337,0,False


In [54]:
df_train = df_train#.head(10000)
df_test = df_test#.head(10000)

In [55]:
X_train = df_train[dict_info["cols_x"] + dict_info["col_pid"]]
y_train = np.ravel(df_train[dict_info["col_y"]])

In [56]:
X_test = df_test[dict_info["cols_x"] + dict_info["col_pid"]]
y_test = np.ravel(df_test[dict_info["col_y"]])

In [57]:
fit_models = True

In [58]:
import joblib

In [59]:
param_grid = {
    "n_estimators": [50, 100], #
    "learning_rate": [0.05, 0.1, 0.15], #
    "max_depth": [2, 3] #
}

model_full = GradientBoostingClassifier()

grid_full = GridSearchCV(
    estimator=model_full,
    param_grid=param_grid,
    cv=5,                # 5-fold cross-validation
    scoring="neg_log_loss",  # or another metric
    n_jobs=-1            # use all CPU cores
)

if fit_models:
    grid_full.fit(X_train, y_train)
    joblib.dump(grid_full, "models/gridsearch-mm.pkl")
else:
    grid_full = joblib.load("models/gridsearch-mm.pkl")

/home/hcai/mirko.thalmann/miniconda3/envs/representationsID/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:114: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/hcai/mirko.thalmann/miniconda3/envs/representationsID/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:114: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/hcai/mirko.thalmann/miniconda3/envs/representationsID/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:114: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)
/home/hcai/mirko.thalmann/miniconda3/envs/representatio

In [60]:
best_model_full = grid_full.best_estimator_
y_test_pred = best_model_full.predict(X_test)
y_train_pred = best_model_full.predict(X_train)
df_train_preds_full = pd.DataFrame({"y_train_true":np.ravel(y_train), "y_train_pred":y_train_pred}) 
df_train_preds_full["is_correct"] = df_train_preds_full["y_train_true"] == df_train_preds_full["y_train_pred"]
df_test_preds_full = pd.DataFrame({"y_test_true":np.ravel(y_test), "y_test_pred":y_test_pred})
df_test_preds_full["is_correct"] = df_test_preds_full["y_test_true"] == df_test_preds_full["y_test_pred"]

In [61]:
print(
    "TIME AND VALUE:\n",
    "train accuracy: ", np.round(df_train_preds_full["is_correct"].mean(), 3), 
    "\ntest accuracy: ", np.round(df_test_preds_full["is_correct"].mean(), 3)
)

TIME AND VALUE:
 train accuracy:  0.697 
test accuracy:  0.693


## check number of participants per trial number

In [74]:
df_response_agg = (
    df.group_by("ResponseID")
    .len("n_scenarios_saved")
    .filter(pl.col("n_scenarios_saved") == 2)
)

# only scenarios with pairs of rows
df_scenario_pairs = df.join(df_response_agg, on="ResponseID")

In [76]:
df_scenario_pairs.group_by("UserID").len("n_trials").group_by("n_trials").len("n_subjects").collect().sort("n_trials")

n_trials,n_subjects
u32,u32
2,55516
4,44212
6,35678
8,28985
10,22450
12,17982
14,15513
16,21457
18,50780


In [77]:
20 * 111899 + 22 * 818577 + 24 * 313391 + 26 * 55052

29199410